Run before everything

In [ ]:
from ugot import ugot  # UGOT robot SDK
from IPython.display import clear_output  # Jupyter display helpers

# Create a robot instance and connect over the network.
got = ugot.UGOT()
got.initialize("192.168.88.1")  # <-- Check if this matches your robot's IP address

# Pre-load vision models onto the UGOT
# All models listed here will be available for the rest of the session.
got.load_models(
    ["color_recognition", "word_recognition", "line_recognition", "apriltag_qrcode"]
)

# Open UGOT camera for later image recognition
got.open_camera()

Part2

In [ ]:
import time

def part2run():
    print("Part 2 Started")
    choice = ""

    # --- 1. Recognise Text (10 pts) ---
    while True:
        # OCR to read the sign held by the judge
        word = str(got.get_words_result()).upper()

        if "LEFT" in word:
            choice = "LEFT"
            got.screen_display_background(6) 
            got.screen_print("LEFT") # Show recognition to judge
            break
        elif "RIGHT" in word:
            choice = "RIGHT"
            got.screen_display_background(6)
            got.screen_print("RIGHT") # Show recognition to judge
            break
        time.sleep(0.2)

    # --- 2. Move to Correct Path (20 pts) ---
    # Nudge the robot toward the chosen fork
    if choice == "LEFT":
        got.mecanum_turn_speed_times(turn=2, speed=30, times=35, unit=2)
    else:
        got.mecanum_turn_speed_times(turn=3, speed=30, times=35, unit=2)

    # --- 3. Line Tracing (30 pts) ---
    # Criterion: Wheels must not cross the line
    got.set_track_recognition_line(0) 
    while True:
        # Follow line using proportional steering (mult)
        offset, line_type, x, y = got.get_single_track_total_info()
        
        if line_type == 0: # End of the path
            got.mecanum_stop()
            got.screen_print("FINISHED")
            break
        else:
            steering = int(offset * 0.28)
            got.mecanum_move_xyz(0, 30, steering)
            
        time.sleep(0.01)

# Run Part 2
try:
    part2run()
except KeyboardInterrupt:
    got.mecanum_stop()